# RTC-SDD — XLS-R 300M + AASIST · P01 Research Pipeline

**Kaggle launcher / reproducibility notebook** · FP32 fine-tuning · saved checkpoint restoration · RTCFake progress inference · Codabench submission.

> **Status:** The first P01 epoch finished successfully. The existing trained model can be restored and used for prediction **without retraining**. Training and full inference are **opt-in** below (disabled by default).
>
> **Source of truth:** Model, training and validation logic live in `src/rtcsdd/` and `scripts/`. This notebook intentionally does not embed 40+ historical patches, manual debugging cells or checkpoint contents.

**Important:** The older copy of the Git repository did not contain all P01 changes. This notebook assumes the **updated** P01 source files and `scripts/progress_inference_p01.py` have been committed alongside it. It will stop with a clear error if those files are missing. See `README.md` and `docs/` for project details.

**Data:** attach the authorized private Kaggle RTCFake pack dataset. Never commit gated audio, private labels, authentication tokens or pretrained model weights.

## 1 — Paths & settings

Configure the project, competition dataset and Hugging Face checkpoint repository once here. We use **Python 3.10** in a dedicated virtual environment, since the pinned Fairseq baseline is sensitive to its dependencies.

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/RTC-SDD-Research")
DATA = Path("/kaggle/input/datasets/a7medlabib/rtcfake-packed-private")
VENV = Path("/kaggle/working/rtcsdd-baseline-py310")
PY310 = VENV / "bin/python"
PRETRAINED = Path("/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt")
OFFICIAL = ROOT / "third_party/official_rtc_sdd"
CONFIG = ROOT / "src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json"
OUT = ROOT / "outputs/experiments/p01_xlsr_aasist_official_like_fp32"

REPO_URL = "https://github.com/AhmedLabib1/RTC-SDD-Research.git"
HF_REPO = "AhmedLabib1/rtc-sdd-research-checkpoints"
HF_REMOTE = "experiments/p01_xlsr_aasist_official_like_fp32"

print("Project:", ROOT)
print("Dataset:", DATA)
print("Experiment:", OUT)

## 2 — Clone the repository (when needed)

If working in the same Kaggle session, the existing checkout is preserved. For a fresh session, this cell clones the repository. If private access is needed, add `GITHUB_TOKEN` to **Kaggle Secrets** first. No token is saved in notebook output or in the repository.

In [ ]:
import os
import subprocess
import tempfile
from kaggle_secrets import UserSecretsClient

if (ROOT / ".git").is_dir():
    print("Using existing checkout:", ROOT)
else:
    assert not ROOT.exists(), f"Destination exists without .git: {ROOT}"
    try:
        github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        github_token = None

    env = os.environ.copy()
    if github_token:
        with tempfile.TemporaryDirectory() as td:
            askpass = Path(td) / "git-askpass.sh"
            askpass.write_text(
                '#!/bin/sh\n'
                'case "$1" in\n'
                '  *Username*) printf "%s" "x-access-token" ;;\n'
                '  *) printf "%s" "$GITHUB_TOKEN" ;;\n'
                'esac\n'
            )
            askpass.chmod(0o700)
            env.update({
                "GITHUB_TOKEN": github_token,
                "GIT_ASKPASS": str(askpass),
                "GIT_TERMINAL_PROMPT": "0",
            })
            subprocess.run(
                ["git", "-c", "credential.helper=", "clone", REPO_URL, str(ROOT)],
                env=env, check=True,
            )
    else:
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    print("Repository cloned successfully")
    del github_token

assert (ROOT / "pyproject.toml").is_file(), "Missing project pyproject.toml"

## 3 — Pinned official baseline & Python 3.10 environment

Restore the pinned official RTC-SDD source (kept outside Git), create the Python 3.10 venv, and install the official training requirements plus our package. The old notebook converted the pinned editable Fairseq requirement to a standard PEP 508 reference for compatibility with `uv`; that conversion is retained here.

This installs dependencies but **does not train**.

In [ ]:
import re
import sys
import shutil
import subprocess

assert shutil.which("uv"), "uv is missing from Kaggle PATH"
if not PY310.is_file():
    subprocess.run(["uv", "python", "install", "3.10"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.10", str(VENV)], check=True)

subprocess.run(
    [sys.executable, str(ROOT / "scripts/setup/bootstrap_official.py")],
    cwd=ROOT, check=True,
)
subprocess.run(
    [sys.executable, str(ROOT / "scripts/setup/install_training_env.py"),
     "--python", str(PY310)],
    cwd=ROOT, check=True,
)

req_file = OFFICIAL / "requirements.txt"
assert req_file.is_file(), req_file
converted = []
matched = 0
for line in req_file.read_text(encoding="utf-8").splitlines():
    match = re.fullmatch(
        r"-e\s+git\+https://github\.com/pytorch/fairseq@"
        r"([0-9a-fA-F]{40})#egg=fairseq", line.strip(),
    )
    if match:
        converted.append(
            f"fairseq @ git+https://github.com/pytorch/fairseq@{match.group(1)}"
        )
        matched += 1
    else:
        converted.append(line)
assert matched == 1, f"Expected one pinned Fairseq reference; got {matched}"
converted_req = Path("/kaggle/working/rtcsdd-requirements-uv.txt")
converted_req.write_text("\n".join(converted) + "\n", encoding="utf-8")

subprocess.run([
    "uv", "pip", "install", "--python", str(PY310),
    "--index-strategy", "unsafe-best-match", "-r", str(converted_req),
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
], check=True)
subprocess.run([
    "uv", "pip", "install", "--python", str(PY310),
    "-e", str(ROOT), "huggingface_hub>=0.30,<1.0",
], check=True)
print("Pinned Python environment ready")

## 4 — Verify GPU, source and dataset

Check CUDA and Fairseq in the *training venv*, then validate that the updated repository actually contains the P01 files. This intentionally replaces temporary P00/v0.4/v0.5/v0.6/v0.7 patch-copying cells; those source changes belong in Git, not in a reproducible notebook.

In [ ]:
import subprocess

required_files = [
    ROOT / "src/rtcsdd/models/xlsr_aasist.py",
    ROOT / "src/rtcsdd/training/xlsr_aasist_finetune.py",
    ROOT / "src/rtcsdd/training/trainer.py",
    ROOT / "src/rtcsdd/training/config.py",
    ROOT / "src/rtcsdd/storage/huggingface.py",
    ROOT / "src/rtcsdd/eval/submission.py",
    ROOT / "scripts/progress_inference_p01.py",
    CONFIG,
    DATA / "train_label.txt",
    DATA / "dev_label.txt",
    DATA / "progress.txt",
    DATA / "progress.pack",
]
missing = [str(p) for p in required_files if not p.is_file()]
assert not missing, "Missing required files (update/commit source first):\n" + "\n".join(missing)

subprocess.run([
    str(PY310), "-c",
    "import numpy as np; "
    "[(setattr(np, k, v)) for k, v in "
    "{'float':float,'int':int,'complex':complex,'bool':bool,'object':object,'str':str}.items() "
    "if k not in np.__dict__]; "
    "import torch, fairseq; "
    "print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available()); "
    "assert torch.cuda.is_available(), 'GPU unavailable'; "
    "print('GPU:', torch.cuda.get_device_name(0))",
], cwd=ROOT, check=True)
print("P01 code, data and CUDA checks passed")

## 5 — Download the original XLS-R 300M pretrained weights

The official XLS-R 300M file is required by the upstream model constructor even when loading the P01 fine-tuned checkpoint afterward. Download it **only if missing**. File size is approximately 3.55 GiB; sha256 is printed to facilitate reproducibility.

In [ ]:
import hashlib
import requests
from tqdm.auto import tqdm

url = "https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr2_300m.pt"
PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
if not PRETRAINED.is_file() or PRETRAINED.stat().st_size < 800_000_000:
    tmp = PRETRAINED.with_suffix(".pt.part")
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with tmp.open("wb") as target:
            for chunk in tqdm(
                response.iter_content(chunk_size=4*1024*1024),
                total=(total+4*1024*1024-1)//(4*1024*1024) if total else None,
                unit="chunk",
            ):
                if chunk:
                    target.write(chunk)
    assert tmp.stat().st_size > 800_000_000
    tmp.replace(PRETRAINED)

digest = hashlib.sha256()
with PRETRAINED.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
print("Pretrained checkpoint:", PRETRAINED)
print("SHA256:", digest.hexdigest())

## 6 — Training (optional; **disabled by default**)

**P01 training already completed one full epoch:** 75,785 training recordings, 4,737 optimizer updates, 15,204 dev recordings. Precision was FP32 (no autocast/GradScaler); micro-batch 2 × gradient accumulation 8 = effective batch size 16. Official RawBoost Algorithm 5 was used for training only.

To reproduce or continue training deliberately, set `TRAIN_MODE` below to `"new"` or `"resume"`. **Do not run this accidentally**: it consumes GPU time. A full resumption requires `last.pt` (optimizer state), not `best_online_f1.pt` (inference weights). If cloud checkpoint backup is enabled by the config, `HF_TOKEN` must be set in Kaggle Secrets.

In [ ]:
import json
import os
import subprocess
from kaggle_secrets import UserSecretsClient

TRAIN_MODE = "off"   # "off" | "new" | "resume"
assert TRAIN_MODE in {"off", "new", "resume"}

if TRAIN_MODE == "off":
    print("Training skipped: using previously trained Epoch 1 checkpoint")
else:
    assert PRETRAINED.is_file() and CONFIG.is_file()
    if TRAIN_MODE == "new":
        assert not OUT.exists(), (
            "Experiment folder already exists. Do not overwrite a trained run."
        )
    else:
        assert (OUT / "last.pt").is_file(), (
            "Full resume checkpoint last.pt is required for --resume"
        )

    cfg = json.loads(CONFIG.read_text(encoding="utf-8"))
    assert cfg["training"]["precision"] == "fp32"
    env = os.environ.copy()
    if cfg.get("checkpoint_backup", {}).get("enabled", False):
        token = UserSecretsClient().get_secret("HF_TOKEN")
        assert token, "HF_TOKEN required because checkpoint backup is enabled"
        env["HF_TOKEN"] = token
        del token

    command = [
        str(PY310), "-u", "-m", "rtcsdd.training.xlsr_aasist_finetune",
        "--root", str(ROOT), "--dataset-root", str(DATA),
        "--checkpoint", str(PRETRAINED), "--config", str(CONFIG),
        "--out", str(OUT), "--epochs", "1", "--max-minutes", "0",
    ]
    if TRAIN_MODE == "resume":
        command.append("--resume")
    print("Training mode:", TRAIN_MODE, flush=True)
    subprocess.run(command, cwd=ROOT, env=env, check=True)

## 7 — Restore the **trained** checkpoint and development metrics

This is the normal path for evaluation or submission after a Kaggle reset. The private Hugging Face model repository holds `best_online_f1.pt`, experiment manifest, and `dev_epoch_001.json`. **Only downloads** happen here; no checkpoint uploads and no retraining.

The selected checkpoint is the best checkpoint for **Online Dev Macro-F1**, not necessarily the best for the hidden leaderboard.

In [ ]:
import json
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
assert hf_token, "Set HF_TOKEN in Kaggle Secrets"
OUT.mkdir(parents=True, exist_ok=True)

for filename in ("best_online_f1.pt", "config.json", "dev_epoch_001.json"):
    target = OUT / filename
    if not target.is_file():
        downloaded = hf_hub_download(
            repo_id=HF_REPO, repo_type="model",
            filename=f"{HF_REMOTE}/{filename}",
            token=hf_token, local_dir=ROOT / "outputs",
        )
        assert Path(downloaded).resolve() == target.resolve()
    print("READY:", filename, f"({target.stat().st_size/1024**2:.2f} MiB)")
del hf_token

results = json.loads((OUT / "dev_epoch_001.json").read_text(encoding="utf-8"))
print("Epoch:", results["epoch"], "| Optimizer step:", results["step"])
print("Online Dev Macro-F1:", results["dev_online_macro_f1"])
print("All-Dev Macro-F1:", results["subgroups"]["all_dev"]["macro_f1"])
print("Weighted dev loss:", results["dev_weighted_loss"])

### Observed Epoch 1 results (historical)

| Metric | Value |
|:--|--:|
| Online Dev Macro-F1 (fixed threshold 0.5) | **0.362177** |
| Offline Dev Macro-F1 | **0.322469** |
| All-Dev Macro-F1 | **0.342422** |
| Weighted Dev Loss | **0.670940** |
| Spoof Recall (All Dev) | **0.241684** |

These are **local Dev metrics**, not official hidden test scores. The model favors bonafide predictions at threshold 0.5; calibration, class weighting, and robustness need further study. Recorded Codabench submissions scored **32.2484** and **45.6397**, but the correspondence between those submissions and model variants must be verified before attributing differences.

## 8 — Validate Progress archive structure

Progress has **30,279 unlabeled recordings**. Unlike ZIP-backed Train/Dev `.pack` files, `progress.pack` is a **TAR.GZ stream**. Archive members begin with `progress/`, but IDs in `progress.txt` begin at `clean/` or `noisy/` and must be kept in official order. We do not extract the full archive to disk.

In [ ]:
import tarfile
import gzip

protocol_ids = [
    item.strip() for item in (DATA / "progress.txt").read_text(encoding="utf-8").splitlines()
    if item.strip()
]
assert len(protocol_ids) == len(set(protocol_ids)) == 30279

with gzip.open(DATA / "progress.pack", "rb") as f:
    tar_header = f.read(512)
assert tar_header[257:262] == b"ustar", "Progress archive is not TAR.GZ"

print("Unique Progress IDs:", len(protocol_ids))
print("First ID:", protocol_ids[0])
print("Archive format: TAR.GZ (streaming)")

## 9 — Full Progress inference (**opt-in**, no Hugging Face uploads)

The implementation lives in [`scripts/progress_inference_p01.py`](../scripts/progress_inference_p01.py) and uses the fine-tuned checkpoint. It reads TAR.GZ sequentially and writes **local** `progress_scores_partial.txt` after each batch, allowing resumption inside a surviving Kaggle session. It uses the Spoof score `softmax(logits)[:, 0]` and generates `submission_progress.txt` in `progress.txt` order.

**No Hugging Face uploading or cloud backup occurs during inference.** A Kaggle session reset can still erase local partial scores. Set `RUN_PROGRESS_INFERENCE=True` to process the full 30,279 recordings; it takes GPU time.

In [ ]:
import subprocess

RUN_PROGRESS_INFERENCE = False  # Opt in only when ready
SCRIPT = ROOT / "scripts/progress_inference_p01.py"
assert SCRIPT.is_file(), "Commit scripts/progress_inference_p01.py with this notebook"

if RUN_PROGRESS_INFERENCE:
    assert (OUT / "best_online_f1.pt").is_file()
    subprocess.run(
        [str(PY310), "-u", str(SCRIPT)], cwd=ROOT, check=True,
    )
    print("Progress inference completed")
else:
    print("Skipped GPU inference. Existing scores were not changed.")

## 10 — Package an official `submission.zip` (**only after inference**)

The final archive must contain exactly **one top-level file** named `scores.txt`, with **30,279** rows in the same order as `progress.txt`. The two whitespace-separated columns are `utterance_id` and **Spoof probability** in `[0,1]`.

Enable the last cell only after `submission_progress.txt` is fully generated. This notebook **does not submit** or upload results anywhere automatically.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import math

BUILD_SUBMISSION_ZIP = False  # Change to True after full inference
source_scores = OUT / "submission_progress.txt"
zip_path = OUT / "submission.zip"

if BUILD_SUBMISSION_ZIP:
    assert source_scores.is_file(), "Complete inference first"
    rows = source_scores.read_text(encoding="utf-8").splitlines()
    ids = [
        s.strip() for s in (DATA / "progress.txt").read_text(encoding="utf-8").splitlines()
        if s.strip()
    ]
    assert len(rows) == len(ids) == 30279
    for uid, row in zip(ids, rows):
        parts = row.split()
        assert len(parts) == 2 and parts[0] == uid
        score = float(parts[1])
        assert math.isfinite(score) and 0 <= score <= 1

    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
        zf.write(source_scores, arcname="scores.txt")
    with ZipFile(zip_path) as zf:
        assert zf.namelist() == ["scores.txt"]
        assert zf.testzip() is None
    print("READY:", zip_path, "| Rows:", len(rows))
else:
    print("Submission packaging skipped (BUILD_SUBMISSION_ZIP=False)")

---
### Reproducibility / safe usage notes

- **Code changes belong in Git**: the initial research notebook copied historical v0.4–v0.7 patches from Kaggle datasets. Those temporary repair cells were deliberately removed. Make sure the final source files in this working tree are committed before relying on a fresh clone.
- **Artifacts do not belong in Git**: RTCFake `.pack`, private labels, `.pt` checkpoints, interim outputs, `submission.zip`, and tokens must stay out of the repository.
- **Do not use Progress labels or leaderboard outcomes to tune the model**. Use only the permitted development split for threshold selection and diagnostics.
- **Avoid accidental reruns**: training, full inference and ZIP packaging are disabled by default above. To continue training from an actual optimizer checkpoint, restore `last.pt`, not the compact model weights.
- **Official RTC-SDD**: https://www.junxue.tech/rtc-sdd-challenge/ · https://github.com/JunXue-tech/RTC-SDD